In [13]:
pip install openpyxl.styles

Defaulting to user installation because normal site-packages is not writeableNote: you may need to restart the kernel to use updated packages.



ERROR: Could not find a version that satisfies the requirement openpyxl.styles (from versions: none)
ERROR: No matching distribution found for openpyxl.styles


In [6]:
pip install kiwipiepy

Defaulting to user installation because normal site-packages is not writeable
Note: you may need to restart the kernel to use updated packages.


In [41]:
from openpyxl import load_workbook
from openpyxl.styles import Font, Alignment
from kiwipiepy import Kiwi
from collections import Counter
import re

src = r"네이버 키워드.xlsx"
out = r"네이버 키워드_result3.xlsx"

# ==== 키워드 합계 & 사이즈 ====

def find_col_by_header(ws, keywords, header_row = 1) :
    # keywords = ["연관키워드", "월간검색수 (PC)", "월간검색수 (MOBILE)", ... , "월평균노출 광고수"]
    for col in range(1, ws.max_column + 1) :
        v = ws.cell(row=header_row, column=col).value
        if not v :
            continue
        s = str(v).replace(" ", "") # 월간검색수 (PC) -> 월간검색수(PC)
        ok = True
        for kw in keywords :
            if kw.replace(" ", "") not in s :
                ok = False
                break
        if ok :
            return col
    return None

wb = load_workbook(src)
ws = wb.active

pc_col = (
    find_col_by_header(ws, ["월간검색수", "PC"])
    or find_col_by_header(ws, ["월간검색수", "피씨"])
    or 2
)
mo_col = (find_col_by_header(ws, ["월간검색수", "모바일"])
          or find_col_by_header(ws, ["월간검색수", "Mobile"])
          or 3
)

total_col = 4
size_col = 5

ws.cell(row=1, column=total_col).value = "전체검색량(PC+모바일)"
ws.cell(row=1, column=size_col).value = "키워드규모(전체검색량)"

for c in [total_col, size_col] :
    cell = ws.cell(row=1, column=c)
    cell.font = Font(bold=True)
    cell.alignment = Alignment(horizontal="center")

# 3.14 -> 실수 (float), 9 10 -> 정수 (int)
# "1,439.34" => int (에러) // int(float()) -> 1439 (가능)

def to_int(x) :
    if x is None :
        return 0
    if isinstance(x, (int, float)) :
        return int(x)
    s = str(x).strip()
    if s in ("<10", "< 10") :
        return 0
    s = s.replace(",", "")
    try :
        return int(float(s))
    except :
        return 0

for r in range(2, ws.max_row + 1) :
    pc = to_int(ws.cell(row=r, column=pc_col).value)
    mo = to_int(ws.cell(row=r, column=mo_col).value)
    total = pc + mo
    ws.cell(row=r, column=total_col).value = total

    if total >= 3000 :
        size = "대형(3000+)"
    elif total >= 1000 :
        size = "중형(1000+)"
    else :
        size = "소형(<1000)"    
    ws.cell(row=r, column=size_col).value = size

ws.auto_filter.ref = f"A1:{ws.cell(row=1, column=max(ws.max_column, size_col)).coordinate}"

# ==== 키워드 형태소 분석 ====

fixed_categort = {
    "원터치": ("타입", "원터치"),
    "드롭": ("타입", "드롭"),
    "클립": ("타입", "클립"),
    "명품": ("타입", "명품"),
    "브랜드": ("타입", "브랜드"),
    "여자": ("타입", "여자")
}
    
brands = {"샤넬", "스톤헨지", "스와로브스키", "티파니", "까르띠에", "불가리", "반클리프", "구찌"}

def classify_token(token: str) :
    str(token).strip()

    if not t :
        return ("기타", "")

    if t in fixed_category :
        return fixed_category[x]

    if t in brands :
        return("브랜드", t)

        return("기타", "")

    
if "키워드_빈도" in wb.sheetnames :
    del wb["키워드_빈도"]
ws_kw = wb.create_sheet("키워드_빈도")

keyword_col = 1
texts = [
    row[0] for row in ws.iter_rows(min_row=2, max_col=1, values_only=True)
    if row and row[0]
]
all_text = " ".join(map(str, texts))

kiwi = Kiwi()
tokens = kiwi.tokenize(all_text)

noun_tags = {"NNG", "NNP", "NNB", "NR"}
noun_tokens = [t.form for t in tokens if t.tag in noun_tags]

regex_tokens = re.findall(r"[A-Za-z]*\d+[A-Za-z]+|\d+[A-Za-z]*", all_text)

stopwords = {"엑스", "구매", "다음", "때문", "생각", "만족"}

valid_one = {"금", "은", "동", "링", "침"}

combined = noun_tokens + regex_tokens
filtered = [t for t in combined if (len(str(t)) >1 or t in valid_one) and t not in stopwords]



count = Counter(filtered)

ws_kw.append(["키워드", "빈도", "카테고리", "세부범주"])

for cell in ws_kw[1] :
    cell.font = Font(bold=True)
    cell.alignment =Alignment(horizontal="centet")
    
for kw, freq in count.most_common() :
    cat, sub = classify_token(kw)
    ws_kw.append([kw, freq, cat, sub])

ws_kw.auto_filter.ref = f"A1:D{ws_kw.max_row}"   


# 14k, gold14k, 18k, 925, k14
# 탐색보고서, 검색창 -> 검색하려고 할 때
# 특정 경로값을 포함한 주소 찾아
# https://www.naver.com/category/shopping?utm_campaign=""&

# ["귀걸이", "진주귀걸이", "진주목걸이"....], tuple ("귀걸이",) -> 인덱스, iter, 반복문, 추가, 삭제, 편집 x
# 귀걸이 진주귀걸이 진주목걸이 20대의악세서리....
# 명사 : 귀걸이, 진주, 목걸이 -> 
# [{"tag": "N", "form": "귀걸이"}, {...}]
# NNG = Noun Noun General : 일반명사 -> 귀걸이, 반지, 목걸이, 디자인
# NNP = Noun Noun Proper : 고유명사 -> 샤넬, 스와로브스키, 티파니
# NNB = Noun Noun Bound : 의존명사 -> 개, 명, 번, 가지, 수
# NR = Noun Numeral : 수사 -> 하나, 둘, 셋, 세, 열

# 데이터 관련 분야 -> 깃허브, 정규표현식 (GA4 & BigQuery)

# wb.save(out)
# print("완료 : ", out)

# 탐색보고서, 검색창 -> 검색하려고 할 때
# 특정 경로값을 포함한 주소 찾아
# http://www.naver.com/category/shopping?utm_campaign=""&

#print(noun_tokens)

# ["귀걸이", "진주귀걸이", "진주목걸이"], tuple ("귀걸이",) -> 인덱스, liter, 반복문, 추가, 삭제, 편집 x
# 귀걸이 진구귀걸이 진주목걸이 20대의악세사리....
# 명사 : 귀걸이, 진주, 목걸이 ->
# [{"tag"}: "N", {"form": "귀걸이"}, {...}]
# NNG = Noun Noun General : 일반 명사 -> 귀걸이, 반지, 목걸이, 디자인
# NNP = Noun Noun proper : 고유 명사 -> 샤넬, 스와로브스키, 티파니
# NNB = Noun Noun Nound : 의존명사 -> 개, 명, 번, 가지, 수
# NR = Noun Numeral : 수사 -> 하나, 둘, 셋 , 세, 열

# 데이터 관련 분야 -> 깃허브, 정규표현식


wb.save(out)
print("완료 : ", out)


ValueError: Value must be one of {'general', 'centerContinuous', 'right', 'left', 'justify', 'fill', 'center', 'distributed'}

In [14]:

from openpyxl import load_workbook
from openpyxl.styles import Font, Alignment
# from kiwipiepy import Kiwi  # 현재 코드에서 사용 안 해서 주석 처리

src = r"네이버 키워드.xlsx"
out = r"네이버 키워드_result.xlsx"


# ==== 키워드 합계 & 사이즈 ====

def find_col_by_header(ws, keywords, header_row=1):
    """
    header_row(기본 1행)에서 'keywords'에 있는 단어들이 모두 포함된 컬럼을 찾아서 column index 반환.
    예: ["월간검색수", "PC"] -> "월간검색수 (PC)" 같은 헤더 컬럼 찾기
    """
    key_norm = [k.replace(" ", "").strip().lower() for k in keywords]

    for col in range(1, ws.max_column + 1):
        v = ws.cell(row=header_row, column=col).value
        if v is None:
            continue

        header = str(v).replace(" ", "").strip().lower()

        ok = True
        for kw in key_norm:
            if kw not in header:
                ok = False
                break

        if ok:
            return col

    return None


def to_int(x):
    """
    셀 값이 숫자/문자/None 섞여있을 때 안전하게 int로 변환.
    "<10", "< 10" 같은 값은 0 처리.
    "1,439" 같은 콤마 포함 숫자도 처리.
    """
    if x is None:
        return 0

    if isinstance(x, (int, float)):
        return int(x)

    s = str(x).strip()
    if not s:
        return 0

    # 특수 케이스
    if s in ("<10", "< 10"):
        return 0

    # 콤마 제거
    s = s.replace(",", "")

    try:
        return int(float(s))
    except:
        return 0


wb = load_workbook(src)
ws = wb.active

# PC / 모바일 컬럼 자동 찾기 (못 찾으면 기본값 2, 3 사용)
pc_col = (
    find_col_by_header(ws, ["월간검색수", "pc"])
    or find_col_by_header(ws, ["월간검색수", "피씨"])
    or 2
)

mo_col = (
    find_col_by_header(ws, ["월간검색수", "mobile"])
    or find_col_by_header(ws, ["월간검색수", "모바일"])
    or 3
)

# 새로 추가할 컬럼 위치
total_col = ws.max_column + 1
size_col = ws.max_column + 2

# 헤더 작성
ws.cell(row=1, column=total_col).value = "전체검색량(PC+모바일)"
ws.cell(row=1, column=size_col).value = "키워드규모(전체 검색량)"

# 헤더 스타일
for c in [total_col, size_col]:
    cell = ws.cell(row=1, column=c)
    cell.font = Font(bold=True)
    cell.alignment = Alignment(horizontal="center", vertical="center")

# 데이터 계산
for r in range(2, ws.max_row + 1):
    pc = to_int(ws.cell(row=r, column=pc_col).value)
    mo = to_int(ws.cell(row=r, column=mo_col).value)
    total = pc + mo

    ws.cell(row=r, column=total_col).value = total

    if total >= 3000:
        size = "대형(3000+)"
    elif total >= 1000:
        size = "중형(1000+)"
    else:
        size = "소형(<1000)"

    ws.cell(row=r, column=size_col).value = size

# 오토필터 범위 설정 (A1 ~ 마지막 컬럼 1행)
last_col = max(ws.max_column, size_col)
ws.auto_filter.ref = f"A1:{ws.cell(row=1, column=last_col).coordinate}"

# ==== 키워드 형태소 분석 ====
if "키워드_빈도" in wb.sheetnames :
    del wb ["키워드_빈도"]

ws_kw = wb.create_chartsheet("키워드_빈도")

keyword_col = 1
texts = [
    row[0] for row in ws.iter_rows(min_row=2, max_col=1, values_only=True) 
    if row and row[0]
]
all_text = " ".join(map(str, texts))


# ["귀걸이", "진주귀걸이", "진주목걸이"], tuple ("귀걸이",) -> 인덱스, liter, 반복문, 추가, 삭제, 편집 x

wb.save(out)
print("완료 :", out)


'C:\\Users\\user\\dataclass_weekday'

In [22]:

from openpyxl import load_workbook
from openpyxl.styles import Font, Alignment
# from kiwipiepy import Kiwi  # 현재 코드에서 사용 안 해서 주석 처리

src = r"네이버 키워드.xlsx"
out = r"네이버 키워드_result.xlsx"


def find_col_by_header(ws, keywords, header_row=1):
    """
    header_row(기본 1행)에서 'keywords'에 있는 단어들이 모두 포함된 컬럼을 찾아서 column index 반환.
    예: ["월간검색수", "PC"] -> "월간검색수 (PC)" 같은 헤더 컬럼 찾기
    """
    key_norm = [k.replace(" ", "").strip().lower() for k in keywords]

    for col in range(1, ws.max_column + 1):
        v = ws.cell(row=header_row, column=col).value
        if v is None:
            continue

        header = str(v).replace(" ", "").strip().lower()

        ok = True
        for kw in key_norm:
            if kw not in header:
                ok = False
                break

        if ok:
            return col

    return None


def to_int(x):
    """
    셀 값이 숫자/문자/None 섞여있을 때 안전하게 int로 변환.
    "<10", "< 10" 같은 값은 0 처리.
    "1,439" 같은 콤마 포함 숫자도 처리.
    """
    if x is None:
        return 0

    if isinstance(x, (int, float)):
        return int(x)

    s = str(x).strip()
    if not s:
        return 0

    # 특수 케이스
    if s in ("<10", "< 10"):
        return 0

    # 콤마 제거
    s = s.replace(",", "")

    try:
        return int(float(s))
    except:
        return 0


wb = load_workbook(src)
ws = wb.active

# PC / 모바일 컬럼 자동 찾기 (못 찾으면 기본값 2, 3 사용)
pc_col = (
    find_col_by_header(ws, ["월간검색수", "pc"])
    or find_col_by_header(ws, ["월간검색수", "피씨"])
    or 2
)

mo_col = (
    find_col_by_header(ws, ["월간검색수", "mobile"])
    or find_col_by_header(ws, ["월간검색수", "모바일"])
    or 3
)

# 새로 추가할 컬럼 위치
total_col = ws.max_column + 1
size_col = ws.max_column + 2

# 헤더 작성
ws.cell(row=1, column=total_col).value = "전체검색량(PC+모바일)"
ws.cell(row=1, column=size_col).value = "키워드규모(전체 검색량)"

# 헤더 스타일
for c in [total_col, size_col]:
    cell = ws.cell(row=1, column=c)
    cell.font = Font(bold=True)
    cell.alignment = Alignment(horizontal="center", vertical="center")

# 데이터 계산
for r in range(2, ws.max_row + 1):
    pc = to_int(ws.cell(row=r, column=pc_col).value)
    mo = to_int(ws.cell(row=r, column=mo_col).value)
    total = pc + mo

    ws.cell(row=r, column=total_col).value = total

    if total >= 3000:
        size = "대형(3000+)"
    elif total >= 1000:
        size = "중형(1000+)"
    else:
        size = "소형(<1000)"

    ws.cell(row=r, column=size_col).value = size

# 오토필터 범위 설정 (A1 ~ 마지막 컬럼 1행)
last_col = max(ws.max_column, size_col)
ws.auto_filter.ref = f"A1:{ws.cell(row=1, column=last_col).coordinate}"

wb.save(out)
print("완료 :", out)



완료 : 네이버 키워드_result.xlsx


In [39]:
from openpyxl import load_workbook
from openpyxl.styles import Font, Alignment
from kiwipiepy import Kiwi
from collections import Counter
import re

src = r"naverkeyword.xlsx"
out = r"naverkeyword_result_v1.xlsx"

# ==== 키워드 합계 & 사이즈 ====

def find_col_by_header(ws, keywords, header_row=1):
    # keywords = ["월간검색수", "PC"] 같은 형태로 들어오면
    # 헤더 문자열에서 해당 키워드들이 모두 포함된 컬럼을 찾아 column index를 반환
    for col in range(1, ws.max_column + 1):
        v = ws.cell(row=header_row, column=col).value
        if not v:
            continue

        s = str(v).replace(" ", "")  # 월간검색수 (PC) -> 월간검색수(PC)

        ok = True
        for kw in keywords:
            if kw.replace(" ", "") not in s:
                ok = False
                break

        if ok:
            return col

    return None


def to_int(x):
    if x is None:
        return 0
    if isinstance(x, (int, float)):
        return int(x)

    s = str(x).strip()
    if s in ("<10", "< 10"):
        return 0

    s = s.replace(",", "")
    try:
        return int(float(s))
    except:
        return 0


wb = load_workbook(src)
ws = wb.active

pc_col = (
    find_col_by_header(ws, ["월간검색수", "PC"])
    or find_col_by_header(ws, ["월간검색수", "피씨"])
    or 2
)

mo_col = (
    find_col_by_header(ws, ["월간검색수", "모바일"])
    or find_col_by_header(ws, ["월간검색수", "Mobile"])
    or 3
)

# ✅ 기존 4,5에 고정하면 덮어쓸 수 있어서 안전하게 마지막 컬럼 뒤에 추가
total_col = ws.max_column + 1
size_col = ws.max_column + 2

ws.cell(row=1, column=total_col).value = "전체검색량(PC+모바일)"
ws.cell(row=1, column=size_col).value = "키워드규모(전체검색량)"

for c in [total_col, size_col]:
    cell = ws.cell(row=1, column=c)
    cell.font = Font(bold=True)
    cell.alignment = Alignment(horizontal="center", vertical="center")

for r in range(2, ws.max_row + 1):
    pc = to_int(ws.cell(row=r, column=pc_col).value)
    mo = to_int(ws.cell(row=r, column=mo_col).value)
    total = pc + mo
    ws.cell(row=r, column=total_col).value = total

    if total >= 3000:
        size = "대형(3000+)"
    elif total >= 1000:
        size = "중형(1000+)"
    else:
        size = "소형(<1000)"

    ws.cell(row=r, column=size_col).value = size

last_col = max(ws.max_column, size_col)
ws.auto_filter.ref = f"A1:{ws.cell(row=1, column=last_col).coordinate}"

# ==== 키워드 형태소 분석 ====

# ✅ fixed_categort 오타 -> fixed_category 로 통일
fixed_category = {
    "원터치": ("타입", "원터치"),
    "드롭": ("타입", "드롭"),
    "클립": ("타입", "클립"),
    "명품": ("타입", "명품"),
    "브랜드": ("타입", "브랜드"),
    "여자": ("타입", "여자"),
}

brands = {"샤넬", "스톤헨지", "스와로브스키", "티파니", "까르띠에", "불가리", "반클리프", "구찌"}


def classify_token(token: str):
    # ✅ token 정리 결과를 t에 저장
    t = str(token).strip()

    if not t:
        return ("기타", "")

    # ✅ 딕셔너리에 있으면 그 값 반환
    if t in fixed_category:
        return fixed_category[t]

    # ✅ 브랜드면 브랜드로 분류
    if t in brands:
        return ("브랜드", t)

    # ✅ 그 외
    return ("기타", "")


# 시트 새로 만들기
if "키워드_빈도" in wb.sheetnames:
    del wb["키워드_빈도"]
ws_kw = wb.create_sheet("키워드_빈도")

texts = [
    row[0]
    for row in ws.iter_rows(min_row=2, max_col=1, values_only=True)
    if row and row[0]
]
all_text = " ".join(map(str, texts))

kiwi = Kiwi()
tokens = kiwi.tokenize(all_text)

noun_tags = {"NNG", "NNP", "NNB", "NR"}
noun_tokens = [tok.form for tok in tokens if tok.tag in noun_tags]

regex_tokens = re.findall(r"[A-Za-z]*\d+[A-Za-z]+|\d+[A-Za-z]*", all_text)

stopwords = {"엑스", "구매", "다음", "때문", "생각", "만족"}
valid_one = {"금", "은", "동", "링", "침"}

combined = noun_tokens + regex_tokens

# ✅ valid_one (공백 X) 사용
filtered = [
    t for t in combined
    if (len(str(t)) > 1 or t in valid_one) and t not in stopwords
]

count = Counter(filtered)

ws_kw.append(["키워드", "빈도", "카테고리", "세부범주"])

for cell in ws_kw[1]:
    cell.font = Font(bold=True)
    # ✅ centet -> center
    cell.alignment = Alignment(horizontal="center", vertical="center")

for kw, freq in count.most_common():
    cat, sub = classify_token(kw)
    ws_kw.append([kw, freq, cat, sub])

ws_kw.auto_filter.ref = f"A1:D{ws_kw.max_row}"

wb.save(out)
print("완료 :", out)


FileNotFoundError: [Errno 2] No such file or directory: 'naverkeyword.xlsx'

In [40]:
from openpyxl import load_workbook
from openpyxl.styles import Font, Alignment
from collections import Counter
import re
import os

src = r"naverkeyword.xlsx"
out = r"naverkeyword_result_v1.xlsx"

def find_col_by_header(ws, keywords, header_row=1):
    for col in range(1, ws.max_column + 1):
        v = ws.cell(row=header_row, column=col).value
        if not v:
            continue
        s = str(v).replace(" ", "")
        ok = True
        for kw in keywords:
            if kw.replace(" ", "") not in s:
                ok = False
                break
        if ok:
            return col
    return None

def to_int(x):
    if x is None:
        return 0
    if isinstance(x, (int, float)):
        return int(x)
    s = str(x).strip()
    if s in ("<10", "< 10"):
        return 0
    s = s.replace(",", "")
    try:
        return int(float(s))
    except:
        return 0

# ====== 0) 파일 존재 체크 ======
if not os.path.exists(src):
    raise FileNotFoundError(f"엑셀 파일을 못 찾았어: {src}\n현재 실행 폴더에 파일이 있는지 확인해줘.")

wb = load_workbook(src)
ws = wb.active

print("✅ Active sheet:", ws.title)
print("✅ rows:", ws.max_row, "cols:", ws.max_column)

# 헤더 샘플 출력
headers = [ws.cell(row=1, column=c).value for c in range(1, ws.max_column + 1)]
print("✅ header row:", headers)

# ====== 1) PC/MO 컬럼 찾기 ======
pc_col = (
    find_col_by_header(ws, ["월간검색수", "PC"])
    or find_col_by_header(ws, ["월간검색수", "피씨"])
)
mo_col = (
    find_col_by_header(ws, ["월간검색수", "모바일"])
    or find_col_by_header(ws, ["월간검색수", "Mobile"])
    or find_col_by_header(ws, ["월간검색수", "MOBILE"])
)

print("✅ detected pc_col:", pc_col, "header:", ws.cell(row=1, column=pc_col).value if pc_col else None)
print("✅ detected mo_col:", mo_col, "header:", ws.cell(row=1, column=mo_col).value if mo_col else None)

# 못 찾으면 여기서 멈추게(기본값 2,3으로 막 쓰면 오히려 더 헷갈림)
if pc_col is None or mo_col is None:
    raise ValueError("PC/Mobile 검색수 컬럼을 헤더에서 못 찾았어.\n"
                     "위에 출력된 header row를 보고 실제 헤더명을 알려주면 그에 맞게 찾아주는 코드로 바로 바꿔줄게.")

# ====== 2) 합계/사이즈 컬럼 추가 ======
total_col = ws.max_column + 1
size_col = ws.max_column + 2

ws.cell(row=1, column=total_col).value = "전체검색량(PC+모바일)"
ws.cell(row=1, column=size_col).value = "키워드규모(전체검색량)"

for c in [total_col, size_col]:
    cell = ws.cell(row=1, column=c)
    cell.font = Font(bold=True)
    cell.alignment = Alignment(horizontal="center", vertical="center")

for r in range(2, ws.max_row + 1):
    pc = to_int(ws.cell(row=r, column=pc_col).value)
    mo = to_int(ws.cell(row=r, column=mo_col).value)
    total = pc + mo
    ws.cell(row=r, column=total_col).value = total

    if total >= 3000:
        size = "대형(3000+)"
    elif total >= 1000:
        size = "중형(1000+)"
    else:
        size = "소형(<1000)"

    ws.cell(row=r, column=size_col).value = size

last_col = max(ws.max_column, size_col)
ws.auto_filter.ref = f"A1:{ws.cell(row=1, column=last_col).coordinate}"

# ====== 3) 형태소 분석(kiwi) - 없으면 스킵 ======
try:
    from kiwipiepy import Kiwi
    kiwi_ok = True
except Exception as e:
    kiwi_ok = False
    print("⚠️ kiwipiepy import 실패 -> 형태소 분석은 건너뜁니다.")
    print("   에러:", repr(e))

fixed_category = {
    "원터치": ("타입", "원터치"),
    "드롭": ("타입", "드롭"),
    "클립": ("타입", "클립"),
    "명품": ("타입", "명품"),
    "브랜드": ("타입", "브랜드"),
    "여자": ("타입", "여자"),
}
brands = {"샤넬", "스톤헨지", "스와로브스키", "티파니", "까르띠에", "불가리", "반클리프", "구찌"}

def classify_token(token: str):
    t = str(token).strip()
    if not t:
        return ("기타", "")
    if t in fixed_category:
        return fixed_category[t]
    if t in brands:
        return ("브랜드", t)
    return ("기타", "")

if kiwi_ok:
    if "키워드_빈도" in wb.sheetnames:
        del wb["키워드_빈도"]
    ws_kw = wb.create_sheet("키워드_빈도")

    texts = [
        row[0]
        for row in ws.iter_rows(min_row=2, max_col=1, values_only=True)
        if row and row[0]
    ]
    all_text = " ".join(map(str, texts))
    print("✅ text length:", len(all_text))

    kiwi = Kiwi()
    tokens = kiwi.tokenize(all_text)

    noun_tags = {"NNG", "NNP", "NNB", "NR"}
    noun_tokens = [tok.form for tok in tokens if tok.tag in noun_tags]
    regex_tokens = re.findall(r"[A-Za-z]*\d+[A-Za-z]+|\d+[A-Za-z]*", all_text)

    stopwords = {"엑스", "구매", "다음", "때문", "생각", "만족"}
    valid_one = {"금", "은", "동", "링", "침"}

    combined = noun_tokens + regex_tokens
    filtered = [t for t in combined if (len(str(t)) > 1 or t in valid_one) and t not in stopwords]

    count = Counter(filtered)

    ws_kw.append(["키워드", "빈도", "카테고리", "세부범주"])
    for cell in ws_kw[1]:
        cell.font = Font(bold=True)
        cell.alignment = Alignment(horizontal="center", vertical="center")

    for kw, freq in count.most_common():
        cat, sub = classify_token(kw)
        ws_kw.append([kw, freq, cat, sub])

    ws_kw.auto_filter.ref = f"A1:D{ws_kw.max_row}"

wb.save(out)
print("✅ 완료 :", out)


FileNotFoundError: 엑셀 파일을 못 찾았어: naverkeyword.xlsx
현재 실행 폴더에 파일이 있는지 확인해줘.